In [ ]:
# GCAR-RFNet: Gated Cross-Attention Representation and Residual Forest Hybrid

import os
import glob
import math
import random
import itertools
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    matthews_corrcoef, cohen_kappa_score, roc_auc_score,
    average_precision_score, confusion_matrix, classification_report
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from imblearn.over_sampling import SMOTE
from statsmodels.stats.contingency_tables import mcnemar

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import shap
import lime
import lime.lime_tabular


# ---------------------------------------------------------------------------
# Reproducibility
# ---------------------------------------------------------------------------
SEED = 508312


def seed_everything(seed=SEED):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


seed_everything(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[INFO] Hardware Device: {DEVICE}")


# ---------------------------------------------------------------------------
# Dataset resolution
# ---------------------------------------------------------------------------
def resolve_file(patterns):
    for pattern in patterns:
        if os.path.exists(pattern):
            return pattern
    if os.path.exists("/kaggle/input"):
        for root, _, files in os.walk("/kaggle/input"):
            for file in files:
                full_path = os.path.join(root, file)
                for pattern in patterns:
                    base = os.path.basename(pattern)
                    if base.lower() == file.lower():
                        return full_path
    return None


df2_candidates = [
    "/kaggle/input/diabetes-health-indicators-dataset/diabetes_binary_5050split_health_indicators_BRFSS2015.csv",
    "/kaggle/input/datasets/alexteboul/diabetes-health-indicators-dataset/diabetes_binary_5050split_health_indicators_BRFSS2015.csv",
    "../input/diabetes-health-indicators-dataset/diabetes_binary_5050split_health_indicators_BRFSS2015.csv",
    "./diabetes_binary_5050split_health_indicators_BRFSS2015.csv"
]

df3_candidates = [
    "/kaggle/input/diabetes-health-indicators-dataset/diabetes_binary_health_indicators_BRFSS2015.csv",
    "/kaggle/input/datasets/alexteboul/diabetes-health-indicators-dataset/diabetes_binary_health_indicators_BRFSS2015.csv",
    "../input/diabetes-health-indicators-dataset/diabetes_binary_health_indicators_BRFSS2015.csv",
    "./diabetes_binary_health_indicators_BRFSS2015.csv"
]

path_df2 = resolve_file(df2_candidates)
path_df3 = resolve_file(df3_candidates)

if path_df2 is None or path_df3 is None:
    raise FileNotFoundError(
        f"Required datasets could not be resolved. Resolved df2: {path_df2} | Resolved df3: {path_df3}. "
        "Please ensure both diabetes_binary_5050split and diabetes_binary_health_indicators are in Kaggle input."
    )

print(f"[INFO] Ingesting df3 (Training Cohort): {path_df3}")
print(f"[INFO] Ingesting df2 (Balanced Benchmark Cohort): {path_df2}")

df3_raw = pd.read_csv(path_df3)
df2_raw = pd.read_csv(path_df2)

df3 = df3_raw.drop_duplicates().reset_index(drop=True)
df2 = df2_raw.drop_duplicates().reset_index(drop=True)

print(f"[INFO] df3 shape after deduplication: {df3.shape}")
print(f"[INFO] df2 shape after deduplication: {df2.shape}")


# ---------------------------------------------------------------------------
# Feature setup
# ---------------------------------------------------------------------------
TARGET = 'Diabetes_binary'

FEATURE_COLS = [
    'PhysHlth', 'BMI', 'MentHlth', 'Age', 'HighBP', 'DiffWalk', 'GenHlth',
    'HeartDiseaseorAttack', 'HighChol', 'Income', 'Stroke', 'HvyAlcoholConsump',
    'PhysActivity', 'Education', 'Smoker', 'Sex', 'NoDocbcCost', 'Veggies',
    'Fruits', 'CholCheck', 'AnyHealthcare'
]

FEATURE_COLS = [col for col in FEATURE_COLS if col in df3.columns and col in df2.columns]

X = df3[FEATURE_COLS].copy()
y = df3[TARGET].astype(int).copy()

df2_X = df2[FEATURE_COLS].copy()
df2_y = df2[TARGET].astype(int).copy()


# ---------------------------------------------------------------------------
# Split FIRST, then scale (fit only on train) to avoid leakage
# ---------------------------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.10, random_state=SEED, stratify=y
)

scaler = StandardScaler()
X_train_scaled_raw = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
df2_X_scaled = scaler.transform(df2_X)

# SMOTE applied on scaled training fold only
print("[INFO] Executing SMOTE oversampling on training fold (strategy=1.0)...")
oversample = SMOTE(sampling_strategy=1.0, random_state=SEED)
X_train_scaled, y_train_res = oversample.fit_resample(X_train_scaled_raw, y_train)
y_train_res = np.array(y_train_res)

print(f"[INFO] Post-SMOTE Training Size: {X_train_scaled.shape[0]} samples (50/50 balance)")


# ---------------------------------------------------------------------------
# GCAR neural subsystem
# ---------------------------------------------------------------------------
class GatedFeatureTokenizer(nn.Module):
    def __init__(self, num_features, embed_dim):
        super().__init__()
        self.weights = nn.Parameter(torch.randn(num_features, embed_dim) * 0.02)
        self.bias = nn.Parameter(torch.zeros(num_features, embed_dim))
        self.gate = nn.Sequential(
            nn.Linear(embed_dim, embed_dim),
            nn.Sigmoid()
        )

    def forward(self, x):
        tokens = x.unsqueeze(-1) * self.weights.unsqueeze(0) + self.bias.unsqueeze(0)
        return tokens * self.gate(tokens)


class GCARNeuralNet(nn.Module):
    def __init__(self, num_features, embed_dim=32, num_heads=4, hidden_dim=64, dropout=0.2):
        super().__init__()
        self.tokenizer = GatedFeatureTokenizer(num_features, embed_dim)
        self.mha = nn.MultiheadAttention(embed_dim=embed_dim, num_heads=num_heads,
                                         batch_first=True, dropout=dropout)
        self.norm1 = nn.LayerNorm(embed_dim)

        flat_dim = num_features * embed_dim
        self.residual_block = nn.Sequential(
            nn.Linear(flat_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.Mish(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim)
        )
        self.skip_proj = nn.Linear(flat_dim, hidden_dim) if flat_dim != hidden_dim else nn.Identity()
        self.mish = nn.Mish()

        self.head = nn.Sequential(
            nn.Linear(hidden_dim, 32),
            nn.BatchNorm1d(32),
            nn.Mish(),
            nn.Dropout(dropout / 2),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        tokens = self.tokenizer(x)
        attn_out, _ = self.mha(tokens, tokens, tokens)
        tokens = self.norm1(tokens + attn_out)

        flat = torch.flatten(tokens, start_dim=1)
        res = self.skip_proj(flat)
        out = self.mish(self.residual_block(flat) + res)
        return self.head(out).squeeze(-1)


class TabularDataset(Dataset):
    def __init__(self, X_data, y_data):
        self.X = torch.tensor(X_data, dtype=torch.float32)
        self.y = torch.tensor(y_data, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


train_dataset = TabularDataset(X_train_scaled, y_train_res)
train_loader = DataLoader(train_dataset, batch_size=512, shuffle=True, drop_last=False)

num_features = X_train_scaled.shape[1]
gcar_nn = GCARNeuralNet(num_features=num_features, embed_dim=32, num_heads=4,
                        hidden_dim=64, dropout=0.2).to(DEVICE)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(gcar_nn.parameters(), lr=1e-3, weight_decay=1e-4)

print("[INFO] Training GCAR Deep Tabular Backbone...")
EPOCHS = 6
for epoch in range(EPOCHS):
    gcar_nn.train()
    total_loss = 0.0
    for bx, by in train_loader:
        bx, by = bx.to(DEVICE), by.to(DEVICE)
        optimizer.zero_grad()
        logits = gcar_nn(bx)
        loss = criterion(logits, by)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * bx.size(0)
    print(f"       Epoch {epoch+1}/{EPOCHS} - Loss: {total_loss / len(train_dataset):.4f}")


# ---------------------------------------------------------------------------
# Random Forest subsystem
# ---------------------------------------------------------------------------
print("[INFO] Training High-Capacity Random Forest Backbone (n_estimators=100)...")
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=SEED,
    class_weight={0: 1, 1: 20},
    n_jobs=-1
)
rf_model.fit(X_train_scaled, y_train_res)


# ---------------------------------------------------------------------------
# Hybrid inference on df2
# ---------------------------------------------------------------------------
print("[INFO] Executing Hybrid Inference on Unseen Benchmark Dataset (df2)...")

gcar_nn.eval()
df2_dataset = TabularDataset(df2_X_scaled, df2_y.values)
df2_loader = DataLoader(df2_dataset, batch_size=1024, shuffle=False)

nn_probs = []
with torch.no_grad():
    for bx, _ in df2_loader:
        bx = bx.to(DEVICE)
        probs = torch.sigmoid(gcar_nn(bx))
        nn_probs.extend(probs.cpu().numpy())
nn_probs = np.array(nn_probs)

rf_probs = rf_model.predict_proba(df2_X_scaled)[:, 1]

hybrid_probs = 0.85 * rf_probs + 0.15 * nn_probs
hybrid_preds = (hybrid_probs >= 0.50).astype(int)


# ---------------------------------------------------------------------------
# Evaluation on df2
# ---------------------------------------------------------------------------
acc = accuracy_score(df2_y, hybrid_preds)
prec = precision_score(df2_y, hybrid_preds)
rec = recall_score(df2_y, hybrid_preds)
f1 = f1_score(df2_y, hybrid_preds)
mcc = matthews_corrcoef(df2_y, hybrid_preds)
kappa = cohen_kappa_score(df2_y, hybrid_preds)
roc_auc = roc_auc_score(df2_y, hybrid_probs)
pr_auc = average_precision_score(df2_y, hybrid_probs)
cm = confusion_matrix(df2_y, hybrid_preds)

print("\n" + "=" * 65)
print("            GCAR-RFNet BENCHMARK PERFORMANCE (df2)              ")
print("=" * 65)
print(f"Accuracy                   : {acc:.4f} ({acc * 100:.2f}%)")
print(f"Precision                  : {prec:.4f}")
print(f"Recall                     : {rec:.4f}")
print(f"F1-Score                   : {f1:.4f}")
print(f"Matthews Corr. Coef. (MCC) : {mcc:.4f}")
print(f"Cohen's Kappa              : {kappa:.4f}")
print(f"ROC-AUC                    : {roc_auc:.4f}")
print(f"PR-AUC                     : {pr_auc:.4f}")
print("=" * 65)
print("\nClassification Report (df2 Unseen Cohort):\n")
print(classification_report(df2_y, hybrid_preds, target_names=['non-diabetic', 'diabetic']))


# ---------------------------------------------------------------------------
# Baseline + McNemar test
# ---------------------------------------------------------------------------
print("[INFO] Fitting Baseline Logistic Regression on Identical SMOTE Folds...")
baseline_lr = LogisticRegression(max_iter=1000, random_state=SEED)
baseline_lr.fit(X_train_scaled, y_train_res)
baseline_preds = baseline_lr.predict(df2_X_scaled)
base_acc = accuracy_score(df2_y, baseline_preds)

y_true_arr = np.array(df2_y)
corr_prop = (hybrid_preds == y_true_arr)
corr_base = (baseline_preds == y_true_arr)

n00 = int(np.sum(corr_prop & corr_base))
n01 = int(np.sum(corr_prop & (~corr_base)))
n10 = int(np.sum((~corr_prop) & corr_base))
n11 = int(np.sum((~corr_prop) & (~corr_base)))

contingency_table = [[n00, n01], [n10, n11]]
mcnemar_res = mcnemar(contingency_table, exact=False, correction=True)

print("=" * 65)
print("              MCNEMAR'S HYPOTHESIS SIGNIFICANCE TEST            ")
print("=" * 65)
print(f"Baseline Accuracy : {base_acc:.4f} | Proposed Hybrid Accuracy : {acc:.4f}")
print(f"Contingency Table [[n00, n01], [n10, n11]]: {contingency_table}")
print(f"Discordant Pairs  : Hybrid Better (n01)={n01:,} | Baseline Better (n10)={n10:,}")
print(f"Chi-Squared Stat  : {mcnemar_res.statistic:.4f}")
print(f"p-value           : {mcnemar_res.pvalue:.6e}")
if mcnemar_res.pvalue < 0.05:
    print("Interpretation    : The proposed GCAR-RFNet architecture is STATISTICALLY")
    print("                    SUPERIOR to the baseline at p < 0.05.")
else:
    print("Interpretation    : No statistically significant difference detected.")
print("=" * 65)


# ---------------------------------------------------------------------------
# Explainability: SHAP + LIME
# ---------------------------------------------------------------------------
print("\nGenerating SHAP Visualizations...")
sample_indices = np.random.RandomState(SEED).choice(len(df2_X), size=300, replace=False)
sample_X = df2_X.iloc[sample_indices]
sample_X_scaled = scaler.transform(sample_X)

explainer_shap = shap.TreeExplainer(rf_model)
shap_vals = explainer_shap.shap_values(sample_X_scaled)

if isinstance(shap_vals, list):
    shap_attr = shap_vals[1]
else:
    shap_attr = shap_vals[:, :, 1] if len(shap_vals.shape) == 3 else shap_vals

plt.figure(figsize=(10, 6))
shap.summary_plot(shap_attr, sample_X, feature_names=FEATURE_COLS, show=False)
plt.title("SHAP Beeswarm Plot - Clinical Risk Attributions (df2)", fontsize=13)
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 6))
shap.summary_plot(shap_attr, sample_X, feature_names=FEATURE_COLS,
                  plot_type="bar", show=False)
plt.title("SHAP Global Feature Importance Bar Plot (df2)", fontsize=13)
plt.tight_layout()
plt.show()

print("Generating LIME Explanation for Clinical Decision Support...")
lime_explainer = lime.lime_tabular.LimeTabularExplainer(
    training_data=X_train_scaled,
    feature_names=FEATURE_COLS,
    class_names=['non-diabetic', 'diabetic'],
    mode='classification',
    random_state=SEED
)

sample_idx = 0
exp = lime_explainer.explain_instance(
    data_row=df2_X_scaled[sample_idx],
    predict_fn=rf_model.predict_proba,
    num_features=10
)
fig = exp.as_pyplot_figure()
plt.title(f"LIME Patient Decision Attribution (Sample #{sample_idx})", fontsize=12)
plt.tight_layout()
plt.show()


# ---------------------------------------------------------------------------
# Confusion matrix visualization
# ---------------------------------------------------------------------------
print("\nRendering Preserved Visualizations from Baseline Implementation...")


def plot_confusion_matrix(cm,
                          target_names,
                          title='Confusion matrix',
                          cmap=None,
                          normalize=True):
    accuracy = np.trace(cm) / np.sum(cm).astype('float')
    misclass = 1 - accuracy

    if cmap is None:
        cmap = plt.get_cmap('Blues')

    plt.figure(figsize=(8, 6))
    plt.imshow(cm, interpolation='nearest', cmap=cmap)
    plt.title(title)
    plt.colorbar()

    if target_names is not None:
        tick_marks = np.arange(len(target_names))
        plt.xticks(tick_marks, target_names, rotation=0)
        plt.yticks(tick_marks, target_names)

    if normalize:
        cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

    thresh = cm.max() / 1.5 if normalize else cm.max() / 2
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        if normalize:
            plt.text(j, i, "{:0.4f}".format(cm[i, j]),
                     horizontalalignment="center",
                     color="white" if cm[i, j] > thresh else "black")
        else:
            plt.text(j, i, "{:,}".format(cm[i, j]),
                     horizontalalignment="center",
                     color="white" if cm[i, j] > thresh else "black")

    plt.tight_layout()
    plt.ylabel('True label')
    plt.xlabel('Predicted label\naccuracy={:0.4f}; misclass={:0.4f}'.format(accuracy, misclass))
    plt.show()


target_names = ['non-diabetic', 'diabetic']
custom_cmap = mpl.colors.LinearSegmentedColormap.from_list('custom', ['white', '#011b7a'])
plot_confusion_matrix(cm, target_names, 'GCAR-RFNet\nConfusion Matrix', custom_cmap, normalize=False)